# Ongoing three-task run: saved fold evidence

This notebook reads progress and hash-verified completed checkpoints. It does not
pool partial folds, compute final metrics, fit models or update the manuscript.
Copy completed checkpoint directories, the root manifest and resolved configuration.
Set `IIVV_RESULTS` to that copied run/snapshot directory.


In [ ]:
import json, os
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image, FileLink

REPO = Path.cwd()
if not (REPO / "run_revision.py").is_file():
    REPO = REPO.parent
assert (
    REPO / "run_revision.py"
).is_file(), "Start Jupyter in the repository or manuscript directory"
import sys

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from revision.data import TASK_PROTOCOL, TASK_TITLES, TASK3_CAPTION


def csv(path, limit=None):
    path = Path(path)
    if not path.is_file():
        display(Markdown(f"Pending/unavailable: `{path.name}`"))
        return pd.DataFrame()
    try:
        table = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        table = pd.DataFrame()
    display(table if limit is None else table.head(limit))
    return table


def images(paths):
    for path in paths:
        display(Markdown(f"**{path.name}**"))
        display(Image(filename=str(path)))
        if "cancer_vs_lr_hr" in str(path):
            display(Markdown(TASK3_CAPTION.replace("\n", " ")))


from revision.data import digest

ROOT = REPO / os.environ.get("IIVV_RESULTS", "results/revision-three-tasks-v3")
manifest = json.loads((ROOT / "run_manifest.json").read_text())
assert manifest["config"].get("task_protocol") == TASK_PROTOCOL, "Obsolete protocol"
assert not manifest.get("synthetic"), "Synthetic fixture"
display({"status": manifest["status"], "protocol": TASK_PROTOCOL})
completed = []
for marker in sorted((ROOT / "fold-checkpoints").glob("*/*/complete.json")):
    saved = json.loads(marker.read_text())
    assert saved["task"] in manifest["config"]["tasks"]
    for name, expected in saved["artifacts"].items():
        relative = Path(name)
        assert (
            not relative.is_absolute() and ".." not in relative.parts
        ), "Unsafe artifact path"
        assert (
            digest(marker.parent / relative) == expected
        ), f"Changed checkpoint: {marker.parent / relative}"
    classes = json.loads((marker.parent / "classes.json").read_text())
    assert classes["task_protocol"] == TASK_PROTOCOL
    completed.append(
        {
            "task": saved["task"],
            "repeat": saved["fold_key"][0],
            "fold": saved["fold_key"][1],
            "seconds": saved["elapsed_seconds"],
            "folder": str(marker.parent),
        }
    )
display(pd.DataFrame(completed))
print(
    f"Completed outer jobs: {len(completed)}/{len(manifest['config']['tasks']) * manifest['config']['outer_folds'] * manifest['config']['outer_repeats']}"
)

## Training-selected combinations and candidate failures in completed folds

In [ ]:
for row in completed:
    folder = Path(row["folder"])
    display(Markdown(f"### {row['task']}, fold {row['fold']}"))
    display(pd.json_normalize(json.loads((folder / "selections.json").read_text())))
    display(pd.DataFrame(json.loads((folder / "candidate_failures.json").read_text())))

## Saved progress from the running folds

In [ ]:
progress = []
for path in sorted((ROOT / "fold-checkpoints").glob("*/*/progress.json")):
    if not (path.parent / "complete.json").exists():
        progress.append(dict(folder=str(path.parent), **json.loads(path.read_text())))
display(pd.json_normalize(progress))